In [ ]:
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

In [ ]:
#using uproot to write generated event into TTree

!pip install uproot
import uproot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
file4 = uproot.open("PythiaEventsBatchTest.root")

tree = file4['pdEventTree']

#read branches into Awkward array, then numpy arrays

E_array = tree['E'].array().to_numpy()
px_array = tree['px'].array().to_numpy()
py_array = tree['py'].array().to_numpy()
pz_array = tree['pz'].array().to_numpy()
eta_array = tree['eta'].array().to_numpy()
phi_array = tree['phi'].array().to_numpy()

In [ ]:
#custom class for jets:
class MJet:
  """
  class for jets, containing 4-momenta and other useful information for jet clustering;
  argument is simply 4-momenta (E,px,py,pz)
  makes use of NumPy for vectorized operations, and other math operations
  """

  def __init__(self, E, px, py, pz):
    self.E = E
    self.px = px
    self.py = py
    self.pz = pz



    self.pT = float(np.sqrt(px**2 + py**2))
    self.p = float(np.sqrt(px**2 + py**2 + pz**2))

    self.phi = float(np.arctan2(self.py, self.px))

    # Calculate mass, add tolerance (1e-6)for floating point issues
    m_squared = self.E**2 - self.p**2
    if m_squared < 0 and np.isclose(m_squared, 0, atol=1e-6): # Treat small negative as zero
        self.m = 0.0
    else:
        self.m = float(np.sqrt(m_squared))

                                                                                                                                                    # Calculate eta, handling edge cases for rapidity
                                                                                                                                                    # maybe also use np.isclose()
    if self.E + self.pz == 0:
        self.eta = -np.inf
    elif self.E - self.pz == 0:
        self.eta = np.inf
    else:
        self.eta = (1/2)*float( np.log( (self.E + self.pz) / (self.E - self.pz) ) )


  def __str__(self):

    output = {
        "E": self.E,
        "px": self.px,
        "py": self.py,
        "pz": self.pz,
        "pT": self.pT,
        "p": self.p,
        "phi": self.phi,
        "eta": self.eta,
        "m": self.m
    }
    return str(output)


  def __add__(self, other):
    return MJet(self.E + other.E, self.px + other.px, self.py + other.py, self.pz + other.pz)

  def __eq__(self, other):
    return (self.E == other.E) and (self.px == other.px) and (self.py == other.py) and (self.pz == other.pz)


In [ ]:
file5 = uproot.open("FinalJets.root")

tree2 = file5['finalJets']

#read branches into Awkward array, then numpy arrays

E_array2 = tree2['E'].array().to_numpy()
px_array2 = tree2['px'].array().to_numpy()
py_array2 = tree2['py'].array().to_numpy()
pz_array2 = tree2['pz'].array().to_numpy()

#get the eta, phi of final clustered jets

finaljet_array = [MJet(E_array2[i], px_array2[i], py_array2[i], pz_array2[i]) for i in range(len(E_array2))]


In [ ]:
pT_array = px_array**2 + py_array**2
pT_array_final = px_array2**2 + py_array2**2

eta_array_final = [j.eta for j in finaljet_array]
phi_array_final = [j.phi for j in finaljet_array]

In [ ]:
#Need to obtain bin widths for plotting in eta,phi plane

d_eta = float('inf')
d_phi = float('inf')


#scan through the initial particles,

for i in range(len(eta_array)):
  for j in range(i+1, len(eta_array)):

    if abs(eta_array[i] - eta_array[j]) < d_eta:
      d_eta = abs(eta_array[i] - eta_array[j])

for i in range(len(phi_array)):
  for j in range(i+1, len(phi_array)):

    if abs(phi_array[i] - phi_array[j]) < d_phi:
      d_phi = abs(phi_array[i] - phi_array[j])

print("d_eta:",d_eta)
print("d_phi:",d_phi)

#then through final clustered jets
for i in range(len(eta_array_final)):

  for j in range(i+1, len(eta_array_final)):

    if abs(eta_array_final[i] - eta_array_final[j]) < d_eta:
      d_eta = abs(eta_array_final[i] - eta_array_final[j])

for i in range(len(phi_array_final)):

  for j in range(i+1, len(phi_array_final)):

    if abs(phi_array_final[i] - phi_array_final[j]) < d_phi:
      d_phi = abs(phi_array_final[i] - phi_array_final[j])

print("recheck:")
print("d_eta:",d_eta)
print("d_phi:",d_phi)



d_eta: 0.0009522739600731356
d_phi: 0.00036998155074940975
recheck:
d_eta: 0.00025131725951466066
d_phi: 0.00036998155074940975


In [ ]:
#looking for any infinities...

count_infinity = 0

for j in finaljet_array:

  print(j.eta)

  if j.eta == float('inf'):
    count_infinity += 1

print(count_infinity)

if count_infinity > 0:
  raise ValueError("Found infinity in eta")


-1.23495441158922
-4.691637647317813
1.271787171468117
-1.7781619182721609
2.433087747964368
1.4587793126115216
-2.35583134214823
2.879324943464059
-0.34127041806638236
2.939859341389529
0.16840599520986946
2.4092167563441684
5.884722706899915
-4.414332350387611
-4.51784408741154
-1.9062190459000397
2.0611212807655335
8.013448698422662
-5.4622203980607456
-5.090665375877428
-0.6808425744045145
-3.7555153470314915
1.970841500717238
-3.81927597955229
1.4772654073231226
-0.4639749389543931
-0.4614375214288487
-1.4658229010199266
1.0202024266761784
1.0974383474888287
-4.102618286965612
-8.491733144950409
3.9303581818859494
-1.6876197875502694
-2.944882176997637
1.8609933734938313
-2.688885301251368
-4.598827210199165
-3.4594138496576616
-4.641859098630205
-7.616680627438191
-8.514232250012876
-1.0579928343339355
-1.453480942824032
1.3423630614911524
0.9439523157316151
8.029246143797694
1.7594693325487094
-4.100335175588748
6.14923077252244
-5.636949218728852
0.15904759648364933
-3.63816543

In [ ]:
#number of bins:

Neta1 = (eta_array.max() - eta_array.min())/(d_eta)
Nphi1 = (phi_array.max() - phi_array.min())/(d_phi)

Neta2 = (max(eta_array_final) - min(eta_array_final))/(d_eta)
Nphi2 = (max(phi_array_final) - min(phi_array_final))/(d_phi)

import math

Neta1 = math.ceil(Neta1)
Nphi1 = math.ceil(Nphi1)

Neta2 = math.ceil(Neta2)
Nphi2 = math.ceil(Nphi2)

print("Number of eta bins in initial particle set:", Neta1)
print("Number of phi bins in initial particle set:", Nphi1)

print("Number of eta bins in final clustered jet set:", Neta2)
print("Number of phi bins in final clustered jet set:", Nphi2)

#final selection (could in principle use different bin sizes and numbers for the two plots,
#                 but I prefer consistency)

Neta = max(Neta1, Neta2)
Nphi = max(Nphi1, Nphi2)

Number of eta bins in initial particle set: 76711
Number of phi bins in initial particle set: 16905
Number of eta bins in final clustered jet set: 70234
Number of phi bins in final clustered jet set: 16844


In [ ]:
eta_bin_edges = np.linspace(eta_array.min(), eta_array.max(), Neta+1)
phi_bin_edges = np.linspace(phi_array.min(), phi_array.max(), Nphi+1)

eta_bin_edges_final = np.linspace(min(eta_array_final), max(eta_array_final), Neta+1)
phi_bin_edges_final = np.linspace(min(phi_array_final), max(phi_array_final), Nphi+1)

#initial particles
z1, xedges1, yedges1 = np.histogram2d(eta_array, phi_array, bins=(eta_bin_edges, phi_bin_edges))

#final clustered jets
#z2, xedges2, yedges2 = np.histogram2d(eta_array_final, phi_array_final, bins=(eta_bin_edges_final, phi_bin_edges_final), weights=pT_array_final)